# 05j-i — Regenerative confirmation support

05j-h ha trovato **zero** coppie nel regime intermedio `[-45, -20) mV`. Questo notebook non addestra un modello: genera uno shard teacher separato e validation-only con 24 coppie causali snapshot-matched mirate a quel buco. Il pilot usa solo seed separati; il piano viene congelato prima della nuova acquisizione e tutte le 48 traiettorie vengono conservate. Usare una sessione Kaggle nuova con persistenza `No persistence` o `Files only`, mai la persistenza delle variabili NEURON.

## 1. Checkout riproducibile e teacher canonico

In [ ]:
import importlib, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
ELM_REPOSITORY='https://github.com/Zagred47/giada.git'; ELM_REF=os.environ.get('HAYFLOW_ELM_REF','main')
TEACHER_REPOSITORY='https://github.com/SelfishGene/neuron_as_deep_net.git'; TEACHER_COMMIT='074c4666300a8ad246601dab179a97a6942f0f29'
ROOT=Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd().resolve(); WORKSPACE=ROOT/'hayflow_workspace'; WORKSPACE.mkdir(parents=True,exist_ok=True)
def run(command,cwd=None): print('+',' '.join(map(str,command)),flush=True); subprocess.run(list(map(str,command)),cwd=cwd,check=True)
override=os.environ.get('HAYFLOW_ELM_REPO'); mounted=[Path(override).expanduser()] if override else []; mounted.extend([Path.cwd(),*Path.cwd().parents])
ELM_REPO=next((p.resolve() for p in mounted if (p/'src'/'hayflow_teacher').is_dir()),None)
if ELM_REPO is None:
    ELM_REPO=WORKSPACE/'elmneuron'
    if not (ELM_REPO/'.git').is_dir(): run(['git','clone',ELM_REPOSITORY,ELM_REPO])
    run(['git','fetch','origin',ELM_REF],cwd=ELM_REPO); run(['git','checkout','--detach','FETCH_HEAD'],cwd=ELM_REPO)
TEACHER_REPO=Path(os.environ.get('HAYFLOW_TEACHER_REPO',ELM_REPO.parent/'neuron_as_deep_net')).expanduser().resolve()
if not (TEACHER_REPO/'.git').is_dir(): run(['git','clone',TEACHER_REPOSITORY,TEACHER_REPO])
run(['git','checkout','--detach',TEACHER_COMMIT],cwd=TEACHER_REPO)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=TEACHER_REPO,text=True).strip()==TEACHER_COMMIT
REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ELM_REPO,text=True).strip(); print({'owned':str(ELM_REPO),'teacher':str(TEACHER_REPO),'revision':REVISION})

## 2. Dipendenze e MOD originali (CPU)

In [ ]:
run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7','numpy','pandas','matplotlib','h5py','pyarrow','pyyaml'])
SIMULATION_DIR=TEACHER_REPO/'L5PC_NEURON_simulation'
if not list(SIMULATION_DIR.rglob('libnrnmech.so')):
    nrnivmodl=shutil.which('nrnivmodl') or str(Path(sys.executable).parent/'nrnivmodl'); run([nrnivmodl,'mods'],cwd=SIMULATION_DIR)
assert list(SIMULATION_DIR.rglob('libnrnmech.so'))
sys.path.insert(0,str(ELM_REPO))
for name in tuple(sys.modules):
    if name=='src.hayflow_teacher' or name.startswith('src.hayflow_teacher.') or name=='src.hayflow_data' or name.startswith('src.hayflow_data.'): sys.modules.pop(name,None)
importlib.invalidate_caches(); print('Teacher compilato; GPU non richiesta.')

## 3. Individuazione sicura degli input

In [ ]:
INPUT_ROOT=Path('/kaggle/input')
def extract_zip_safely(source,destination):
    source,destination=Path(source),Path(destination); destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(source) as archive:
        for member in archive.infolist():
            target=(destination/member.filename).resolve(); assert target.is_relative_to(destination.resolve()),member.filename
        archive.extractall(destination)
    return destination
def valid_base(path):
    path=Path(path); return (path/'transition_dataset.h5').is_file() and (path/'targeted_pilot'/'candidate_trials.parquet').is_file() and (path/'artifact_index.json').is_file()
base_candidates=([Path(os.environ['HAYFLOW_BASE_DATASET']).expanduser()] if os.environ.get('HAYFLOW_BASE_DATASET') else [])
if INPUT_ROOT.is_dir(): base_candidates += [p.parent.parent for p in INPUT_ROOT.rglob('candidate_trials.parquet') if p.parent.name=='targeted_pilot']
BASE_DATASET=next((p.resolve() for p in base_candidates if valid_base(p)),None)
if BASE_DATASET is None and INPUT_ROOT.is_dir():
    archives=[p for p in INPUT_ROOT.rglob('*.zip') if 'targeted-transition-dataset' in str(p).lower() or 'targeted_transition_dataset' in p.name.lower()]
    assert len(archives)==1,f'Archivio base ambiguo o assente: {archives}'
    extracted=extract_zip_safely(archives[0],'/kaggle/temp/hayflow05ji_base'); BASE_DATASET=next((p.parent.parent for p in extracted.rglob('candidate_trials.parquet') if valid_base(p.parent.parent)),None)
assert BASE_DATASET is not None,'Base v1.1 completa con candidate_trials.parquet non trovata.'
calibration_candidates=([Path(os.environ['HAYFLOW_CALIBRATION_SOURCE']).expanduser()] if os.environ.get('HAYFLOW_CALIBRATION_SOURCE') else [])
calibration_candidates += [Path('/kaggle/input/datasets/alessandrobelli/hayflow-dendritic-protocol-calibration/hayflow_dendritic_protocol_calibration'),Path('/kaggle/input/hayflow-dendritic-protocol-calibration/hayflow_dendritic_protocol_calibration')]
if INPUT_ROOT.is_dir(): calibration_candidates += [p.parent for p in INPUT_ROOT.rglob('selected_dendritic_protocols.json')]
CALIBRATION_SOURCE=next((p.resolve() for p in calibration_candidates if (p/'selected_dendritic_protocols.json').is_file()),None)
if CALIBRATION_SOURCE is None and INPUT_ROOT.is_dir():
    calibration_archives=[]
    for archive_path in INPUT_ROOT.rglob('*.zip'):
        try:
            with zipfile.ZipFile(archive_path) as archive: names=archive.namelist()
            if any(name.endswith('selected_dendritic_protocols.json') for name in names): calibration_archives.append(archive_path.resolve())
        except zipfile.BadZipFile: pass
    if len(calibration_archives)>1: raise RuntimeError(f'Più archivi di calibrazione trovati: {calibration_archives}')
    if len(calibration_archives)==1:
        extracted_calibration=extract_zip_safely(calibration_archives[0],'/kaggle/temp/hayflow05ji_calibration')
        CALIBRATION_SOURCE=next((p.parent.resolve() for p in extracted_calibration.rglob('selected_dendritic_protocols.json')),None)
if CALIBRATION_SOURCE is None:
    mounted=[str(p) for p in sorted(INPUT_ROOT.glob('*'))] if INPUT_ROOT.is_dir() else []
    raise RuntimeError('Calibrazione 01b non montata. In Add Input aggiungi il dataset hayflow-dendritic-protocol-calibration (owner alessandrobelli), poi riavvia questa cella. Input visibili: '+repr(mounted))
from src.hayflow_teacher.regenerative_confirmation_support import EXPECTED_05JH_INDEX_SHA256
def exact_05jh(path):
    path=Path(path)
    if path.is_file(): return path.name=='hayflow_hines_regenerative_support_expansion.zip'
    for idx in path.rglob('artifact_index.json'):
        import hashlib
        if hashlib.sha256(idx.read_bytes()).hexdigest()==EXPECTED_05JH_INDEX_SHA256: return True
    return False
artifact_candidates=([Path(os.environ['HAYFLOW_05JH_ARTIFACT']).expanduser()] if os.environ.get('HAYFLOW_05JH_ARTIFACT') else [])
if INPUT_ROOT.is_dir(): artifact_candidates += list(INPUT_ROOT.rglob('hayflow_hines_regenerative_support_expansion.zip')) + [p.parent for p in INPUT_ROOT.rglob('artifact_index.json')]
ARTIFACT_05JH_SOURCE=next((p.resolve() for p in artifact_candidates if p.exists() and exact_05jh(p)),None); assert ARTIFACT_05JH_SOURCE is not None,'Artefatto 05j-h esatto non trovato.'
print({'base':str(BASE_DATASET),'calibration':str(CALIBRATION_SOURCE),'05j-h':str(ARTIFACT_05JH_SOURCE)})

## 4. Teacher, base immutabile e provenienza 05j-h

In [ ]:
import yaml
from IPython.display import display
from src.hayflow_teacher import RegenerativeConfirmationConfig, RegenerativeConfirmationSupportSession, expected_audit_hashes
TARGET_CONFIG_PATH=ELM_REPO/'configs'/'hayflow'/'targeted_transition_dataset_v1_1.yml'
CONFIG_PATH=ELM_REPO/'configs'/'hayflow'/'hayflow_regenerative_confirmation_support.yml'
payload=yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8')); confirmation_config=RegenerativeConfirmationConfig.from_mapping(payload['regenerative_confirmation_support'])
target_config=yaml.safe_load(TARGET_CONFIG_PATH.read_text(encoding='utf-8'))
OUTPUT_DIR=Path(os.environ.get('HAYFLOW_OUTPUT_DIR',ROOT/'artifacts'/'hayflow_regenerative_confirmation_support')).expanduser().resolve()
if OUTPUT_DIR.exists(): raise RuntimeError(f'Output protetto già presente: {OUTPUT_DIR}. Usa una nuova HAYFLOW_OUTPUT_DIR o una sessione nuova.')
session=RegenerativeConfirmationSupportSession(ELM_REPO,TEACHER_REPO,base_dataset=BASE_DATASET,calibration_source=CALIBRATION_SOURCE,dataset_config_path=TARGET_CONFIG_PATH,output_dir=OUTPUT_DIR,seed=271828,expected_teacher_hashes=expected_audit_hashes(),native_snapshot_stride=target_config['storage']['native_snapshot_stride_ms'],artifact_05jh_source=ARTIFACT_05JH_SOURCE,confirmation_config=confirmation_config)
teacher_report=session.prepare_teacher(); contract_report=session.prepare_targeted_contract(); base_report=session.verify_base_dataset(verify_large_hdf=True); equilibrium_report=session.import_base_equilibrium(); provenance_report=session.verify_05jh_artifact()
display({'teacher':teacher_report,'contract':contract_report,'base':base_report,'equilibrium':equilibrium_report,'05j-h':provenance_report}); assert teacher_report['segment_count']==642 and base_report['valid'] and provenance_report['valid']

## 5. Pilot breve sui bordi da 1 ms
Qui vengono provati solo seed di pilot separati. Se nessun template autentico entra robustamente nella banda mancante, il notebook si ferma **prima** della generazione lunga e lascia il report diagnostico.

In [ ]:
templates=session.load_source_templates(); pilot_report=session.run_boundary_template_pilot(templates)
display({'source_candidates':len(templates),'pilot_trials':pilot_report['trial_count'],'eligible_templates':pilot_report['eligible_template_count'],'selected_templates':pilot_report['selected_templates']}); assert pilot_report['valid'] and pilot_report['canonical_weight_scaling_used'] is False

## 6. Congelamento del piano e snapshot bank

In [ ]:
protocols,plan=session.build_confirmation_plan(); snapshots=session.prepare_snapshot_bank(protocols,conditioning_ms=confirmation_config.conditioning_ms)
display({'plan':plan,'snapshot_count':snapshots['snapshot_count']}); assert plan['pair_count']==24 and plan['episode_count']==48 and snapshots['snapshot_count']==24

## 7. Generazione completa dello shard

In [ ]:
manifest=session.generate_confirmation_shard(protocols); display({'dataset_role':manifest['dataset_role'],'trajectory_count':manifest['trajectory_count'],'transition_count':manifest['transition_count']}); assert manifest['trajectory_count']==48

## 8. Replay esaustivo e supporto realizzato
La validazione conserva tutti i casi. `valid=True` certifica integrità e replay; `scientific_support_sufficient` dice separatamente se almeno 18 coppie sono realmente nella banda near-regenerative.

In [ ]:
final_report=session.validate_confirmation_shard(protocols); display({'valid':final_report['valid'],'support_sufficient':final_report['scientific_support_sufficient'],'diagnosis':final_report['diagnosis'],'strata':final_report['realized_stratum_counts'],'replay':final_report['exhaustive_replay'],'next_step':final_report['next_step']}); assert final_report['valid'] and final_report['all_registered_episodes_retained'] and not final_report['methodology']['candidate_training_performed']

## 9. Scarica lo ZIP con il metodo browser Blob

In [ ]:
from pathlib import Path
import base64, shutil
from IPython.display import Javascript, display
zip_base=Path('/kaggle/working/hayflow_regenerative_confirmation_support')
zip_path=Path(shutil.make_archive(str(zip_base),'zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name))
encoded=base64.b64encode(zip_path.read_bytes()).decode('ascii'); filename=zip_path.name
display(Javascript(f"""
const b64 = '{encoded}';
const bytes = new Uint8Array(atob(b64).split('').map(c => c.charCodeAt(0)));
const blob = new Blob([bytes], {{type: 'application/zip'}});
const url = URL.createObjectURL(blob);
const a = document.createElement('a'); a.href = url; a.download = '{filename}';
document.body.appendChild(a); a.click(); a.remove();
setTimeout(() => URL.revokeObjectURL(url), 60000);
""")); print({'zip':str(zip_path),'size_mib':round(zip_path.stat().st_size/2**20,1)})